# 📦 Template 1 : préparation des données — AT&T Spam Detector

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Rempli le 19/10/2026 : les cellules ✍️ contiennent la solution.*

## Les 4 templates du projet

| # | Phase | Entrée | Sortie |
|---|---|---|---|
| **1** | **Préparation des données** (ce notebook) | `src/spam.csv` | `src/train.csv`, `src/val.csv`, `src/test.csv` |
| 2 | Modèle simple : embeddings + petit réseau (M05-D05) | les 3 CSV | un modèle + ses scores |
| 3 | Transfer learning : modèle Hugging Face (M06-D02) | les 3 CSV | un modèle + ses scores |
| 4 | Évaluation et conclusion | les scores | le tableau final du livrable |

Les CSV de sortie servent de point de départ aux templates 2 et 3 : ce notebook-ci n'est à faire qu'**une fois**.

## Comment s'en servir

Chaque étape suit le même rythme :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants, et **où tu les as vues** ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie tes chiffres. Lance-la : ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés (`df`, `df_clean`, `train_df`...) sont ceux qu'attendent les contrôles.

## La colonne « Vu où » des tableaux d'outils

- **📍 M02-D01 Basic Data Manipulation** : l'outil est dans cette leçon. Ouvre-la et cherche son nom (Ctrl+F) pour retrouver l'exemple du cours, puis adapte-le.
- **🐍 Python** : une base de Python (`len`, `in`, `.split()`...), pas une fonction pandas.
- **🆕 Hors cours** : l'outil n'apparaît dans aucune leçon. Il est expliqué sur place, avec la raison pour laquelle on en a besoin. Il y en a très peu.

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

DOSSIER_SRC = Path("src")       # le dossier où se trouve spam.csv (à côté de ce notebook)
SEED = 42                       # graine du hasard, la même partout
pd.set_option("display.max_colwidth", None)   # afficher les SMS en entier au lieu de les couper par « ... »
print("prêt")

---
# Étape 1 — Charger le fichier

🎯 Lire `src/spam.csv` dans un DataFrame `df`.

⚠️ Le fichier n'est **pas** en UTF-8 (l'encodage par défaut de pandas) : sans le bon encodage, `read_csv` plante avec `UnicodeDecodeError`. Il est en **latin-1**.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `pd.read_csv(chemin)` | lit un CSV dans un DataFrame | — | 📍 M02-D01 Basic Data Manipulation |
| `encoding="latin-1"` | dit à pandas avec quel « dictionnaire » traduire les octets en lettres | — | 🆕 hors cours : sans lui, ce fichier précis plante |
| `.head()`, `.info()` | premier coup d'œil | — | 📍 M02-D01 Basic Data Manipulation |
| `.shape` | (nombre de lignes, nombre de colonnes) | — | 📍 M02-D01 Data Generation, M02-D02 |

In [ ]:
df = pd.read_csv(DOSSIER_SRC / "spam.csv", encoding="latin-1")
display(df.head())
print(df.shape)
df.info()

👀 **Observe** : combien de colonnes ? Que contiennent `v1` et `v2` ? Combien de valeurs non vides dans les 3 colonnes `Unnamed` ?

In [ ]:
# ✅ Contrôle
assert df.shape == (5572, 5), f"forme attendue (5572, 5), obtenue {df.shape}"
assert list(df.columns[:2]) == ["v1", "v2"]
print("✅ étape 1")

---
# Étape 2 — Nettoyer

## 2.a Repérer les messages coupés

🎯 Les colonnes `Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4` sont presque vides. Mais quand elles ne le sont pas, ce n'est **pas** du bruit : c'est **la fin d'un SMS**. Certains messages contiennent des guillemets mal fermés, et le CSV les a coupés à chaque virgule.

```
v1     v2                                   Unnamed: 2       Unnamed: 3
spam   "Your free ringtone ... FML"          " PO Box 5249"   " MK17 92H. 450Ppw 16\""
       └─────────── un seul SMS, coupé en 3 morceaux à chaque virgule ───────────┘
```

Si tu supprimes ces colonnes tout de suite, tu perds la fin de ces messages.

Les morceaux remplissent les colonnes **dans l'ordre** : un SMS coupé a toujours au moins son 2e morceau dans `Unnamed: 2`. Pour repérer les SMS coupés, il suffit donc de regarder **cette seule colonne**.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `.isnull()` | `True` là où la case est vide | — | 📍 M02-D02 Data Transformation (`events.isnull().sum()`) |
| `.notnull()` | le contraire : `True` là où il y a une valeur | — | 🆕 le jumeau de `.isnull()`, même usage |
| `.sum()` | compte les `True` (un `True` vaut 1) | — | 📍 M02-D02, même cellule |
| `df[masque]` | garde les lignes où le masque vaut `True` | — | 📍 M02-D02 Data Transformation (filtrer avec une condition) |

In [ ]:
masque_coupes = df["Unnamed: 2"].notnull()
print(f"{masque_coupes.sum()} messages coupés")
df[masque_coupes].head(3)

In [ ]:
# ✅ Contrôle
assert masque_coupes.sum() == 50, f"50 messages coupés attendus, obtenu {masque_coupes.sum()}"
print("✅ étape 2.a")

## 2.b Recoller les morceaux

🎯 Créer une colonne `text` = `v2` + les morceaux des colonnes `Unnamed`, **recollés avec une virgule** (la virgule que le CSV a prise pour un séparateur). Pour les 5 522 messages entiers, `text` = `v2`.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `def ma_fonction(row): ... return ...` | une fonction qui reçoit **une ligne** du DataFrame | `row["nom_colonne"]` pour lire une case | 📍 M02-D01 Basic Data Manipulation (`def flag_viral(row)`) |
| `df.apply(ma_fonction, axis=1)` | appelle la fonction sur **chaque ligne** et range les résultats dans une colonne | `axis=1` = ligne par ligne | 📍 M02-D01 (`posts_df.apply(flag_viral, axis=1)`) |
| `.dropna()` | enlève les cases vides | — | 📍 M02-D02 Data Transformation |
| `",".join(morceaux)` | colle des textes avec une virgule entre eux | les morceaux doivent être des **str** (pas de `NaN`) | 📍 M06-D01 Build a transformer : `decode` fait `"".join(...)` (colle sans rien entre) |

C'est le même geste que `flag_viral` dans M02-D01 : une fonction qui regarde une ligne et renvoie une valeur, appliquée à toutes les lignes. Ici, la valeur renvoyée est le texte recollé.

Démo de `.dropna()` + `join` **sur une seule ligne**, dans un autre contexte (une adresse coupée) :

In [ ]:
adresse = pd.Series(["12 rue des Lilas", " Nantes", None, None])   # une ligne avec des cases vides
print(adresse.dropna().tolist())          # les cases vides ont disparu
print(",".join(adresse.dropna()))         # recollé avec une virgule

Chez toi, c'est pareil, mais pour les 5 572 lignes : ce geste va dans une fonction, et `apply` l'appelle sur chaque ligne.

In [ ]:
cols_texte = ["v2", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]

def recoller(row):
    morceaux = row[cols_texte].dropna()
    return ",".join(morceaux)

df["text"] = df.apply(recoller, axis=1)
print(df.loc[masque_coupes, "text"].iloc[0])

In [ ]:
# ✅ Contrôle
exemple = df.loc[df["v2"].str.startswith("Your free ringtone is waiting"), "text"].iloc[0]
assert exemple.endswith('MK17 92H. 450Ppw 16"'), f"la fin du message manque : {exemple!r}"
assert (df.loc[~masque_coupes, "text"] == df.loc[~masque_coupes, "v2"]).all(), "les messages entiers doivent rester identiques à v2"
print("✅ étape 2.b")

## 2.c Garder deux colonnes propres

🎯 Fabriquer `df_clean` avec **deux colonnes seulement** :
- `text` : le SMS recollé ;
- `label` : **0** pour `ham`, **1** pour `spam` (le modèle veut des nombres ; 1 = la classe qu'on cherche).

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| un dictionnaire `{ancienne: nouvelle, ...}` puis `.map(dictionnaire)` | remplace chaque valeur via le dictionnaire | les clés doivent couvrir **toutes** les valeurs, sinon `NaN` | 📍 M02-D01 Basic Data Manipulation (`account_type_mapping` puis `.map(account_type_mapping)`) |
| `df[[col1, col2]]` | garder plusieurs colonnes (double crochet = liste) | — | 📍 M02-D01 (`posts_df[preview_columns]`) |
| `.copy()` | nouveau DataFrame indépendant de `df` | — | 📍 M02-D02 Data Transformation |

In [ ]:
label_mapping = {"ham": 0, "spam": 1}
df["label"] = df["v1"].map(label_mapping)
df_clean = df[["text", "label"]].copy()
display(df_clean.head())
print(df_clean["label"].value_counts())

In [ ]:
# ✅ Contrôle
assert list(df_clean.columns) == ["text", "label"], f"colonnes obtenues : {list(df_clean.columns)}"
assert df_clean["label"].isna().sum() == 0, "des NaN dans label : la traduction a raté une valeur"
assert df_clean["label"].sum() == 747, f"747 spams attendus avant dédoublonnage, obtenu {df_clean['label'].sum()}"
print("✅ étape 2.c")

## 2.d Supprimer les doublons

🎯 Le même SMS apparaît plusieurs fois (spams envoyés en masse, « Ok lar... »). On garde **un exemplaire** de chaque texte.

❓ Question du matin : pourquoi le faire **avant** de séparer train et test ? Pense à ce qui arrive si un même SMS tombe dans les deux : le modèle passe l'examen avec une question qu'il a déjà vue en révision.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `.duplicated(subset=...)` | `True` pour chaque ligne déjà vue plus haut | `subset="text"` : on compare seulement le texte | 📍 M02-D01 Basic Data Manipulation, M02-D02 Data Pipeline |
| `.drop_duplicates(subset=...)` | garde la première, supprime les suivantes | `subset="text"` | 📍 M02-D01 |
| `.reset_index(drop=True)` | renumérote 0, 1, 2... | `drop=True` : jette l'ancien index | 📍 M04-D06 Time Series |

In [ ]:
print(f"{df_clean.duplicated(subset='text').sum()} doublons")
df_clean = df_clean.drop_duplicates(subset="text").reset_index(drop=True)
print(df_clean.shape, f"{df_clean['label'].sum()} spams")

In [ ]:
# ✅ Contrôle
assert df_clean.shape == (5169, 2), f"forme attendue (5169, 2), obtenue {df_clean.shape}"
assert df_clean["label"].sum() == 653, f"653 spams attendus, obtenu {df_clean['label'].sum()}"
assert df_clean.index.tolist() == list(range(5169)), "index pas renuméroté"
print("✅ étape 2.d")

---
# Étape 3 — Explorer

## 3.a Répartition des classes

🎯 Quelle part de spams ? C'est elle qui dit si l'accuracy est une bonne métrique ici.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `.value_counts()` | nombre de lignes pour chaque valeur | — | 📍 M02-D01 Basic Data Manipulation |
| `len(df)` | nombre total de lignes | — | 🐍 Python |
| `px.histogram(df, x=...)` | diagramme en barres d'une colonne | `x="label"` | 📍 M04-D02 Linear regression template |

La part de chaque classe = son nombre de lignes ÷ le nombre total de lignes.

In [ ]:
comptes = df_clean["label"].value_counts()
print(comptes)
print(comptes / len(df_clean))
px.histogram(df_clean, x="label", title="Répartition ham (0) / spam (1)")

In [ ]:
# ✅ Contrôle
assert round(df_clean["label"].mean(), 4) == 0.1263
print("✅ étape 3.a")

✍️ **À écrire (une phrase)** : un « modèle » qui répond **toujours ham** aurait quelle accuracy ? Qu'est-ce que ça dit de l'accuracy comme métrique pour ce projet ?

*Ta réponse :* un modèle qui répond toujours « ham » aurait **87,4 %** d'accuracy (1 − 12,63 %) sans détecter un seul spam : l'accuracy est trompeuse ici, on suivra surtout le **recall** et le **F1** de la classe spam.

## 3.b Longueur des messages

🎯 Comparer la longueur des ham et des spams, en caractères et en mots. Ça servira aussi en phase 2 pour choisir la longueur des séquences.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `len(texte)` | nombre de caractères d'un texte | — | 🐍 Python |
| `texte.split()` | découpe un texte aux espaces → liste de mots | — | 🐍 Python |
| `.map(fonction)` | appelle la fonction sur **chaque case** d'une colonne | la fonction, **sans** parenthèses | 📍 M02-D01 Basic Data Manipulation (`.map(engagement_tier_from_likes)`) |
| `.groupby("label")[cols].median()` | médiane par classe | — | 📍 M02-D02 Data Transformation (`groupby`) |
| `px.histogram(df, x=..., color=...)` | un histogramme par classe, chacun sa couleur | `color="label"` | 📍 M04-D02 (histogram) + 🆕 l'argument `color=` |

`.map` marche avec un dictionnaire (étape 2.c) **ou** avec une fonction (ici) : la fonction reçoit une case, ce qu'elle renvoie va dans la nouvelle colonne. `len` est déjà une fonction. Pour les mots, écris une petite fonction `def nombre_de_mots(texte):` qui renvoie la longueur de la liste de mots.

In [ ]:
df_clean["n_car"] = df_clean["text"].map(len)

def nombre_de_mots(texte):
    return len(texte.split())

df_clean["n_mots"] = df_clean["text"].map(nombre_de_mots)
display(df_clean.groupby("label")[["n_car", "n_mots"]].median())
px.histogram(df_clean, x="n_mots", color="label", title="Nombre de mots par SMS")

In [ ]:
# ✅ Contrôle
med = df_clean.groupby("label")[["n_car", "n_mots"]].median()
assert med.loc[0, "n_car"] == 53 and med.loc[1, "n_car"] == 149, med
assert med.loc[0, "n_mots"] == 11 and med.loc[1, "n_mots"] == 25, med
print("✅ étape 3.b")

## 3.c Combien de mots garder par message ? (préparation de la phase 2)

🎯 En phase 2, chaque SMS deviendra une suite de numéros de **même longueur** (on coupe les trop longs, on complète les trop courts). Choisir une longueur `max_len` qui couvre presque tous les messages, sans s'aligner sur le plus long.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `.describe()` | résumé d'une colonne : moyenne, min, quartiles (25 %, 50 %, 75 %), max | — | 📍 M02-D01 Basic Data Manipulation |
| `.describe(percentiles=[0.95])` | ajoute la ligne « 95 % » : 95 % des messages ont moins de mots que cette valeur | `percentiles=[0.95]` | 🆕 l'argument `percentiles` (le 75 % de `describe` ne suffit pas ici) |

Un centile, c'est la même idée qu'un quartile, en plus fin : le quartile 75 % laisse 25 % des messages au-dessus, le centile 95 % n'en laisse que 5 %.

In [ ]:
display(df_clean["n_mots"].describe())
display(df_clean["n_mots"].describe(percentiles=[0.95]))

In [ ]:
# ✅ Contrôle
assert df_clean["n_mots"].max() == 171
assert round(df_clean["n_mots"].quantile(0.95)) == 33
print("✅ étape 3.c")

✍️ **À écrire** : quelle valeur de `max_len` choisis-tu, et pourquoi pas 171 ?

*Ta réponse :* **`max_len = 40`**. Ça couvre plus de 95 % des messages (95 % = 33 mots dans le `describe`) avec un peu de marge. 171 obligerait à compléter presque tous les SMS avec des zéros (médiane de 11 mots chez les ham) pour un seul message très long : calculs inutiles et signal dilué.

## 3.d Lire les messages

🎯 Lire de vrais exemples : les chiffres ne disent pas tout. Repère ce qui distingue un spam (mots, chiffres, majuscules, numéros courts...).

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `df[df["label"] == 1]` | garder seulement les spams | — | 📍 M02-D02 Data Transformation (filtrer avec une condition) |
| `.sample(n, random_state=...)` | n lignes au hasard, reproductibles | `random_state=SEED` | 📍 M02-D04 Probability Distributions |
| `"motif" in texte` | `True` si le texte contient le motif | — | 🐍 Python |
| `.map(fonction)` puis `.sum()` | appliquer une fonction qui renvoie `True` / `False`, puis compter les `True` | — | 📍 M02-D01 (`map`) + M02-D02 (`sum`) |

Les SMS s'affichent en entier grâce au réglage `display.max_colwidth` de l'étape 0.

In [ ]:
df_hams = df_clean[df_clean["label"] == 0]
df_spams = df_clean[df_clean["label"] == 1]
display(df_hams.sample(5, random_state=SEED)[["text"]])
display(df_spams.sample(5, random_state=SEED)[["text"]])

def contient_lt(texte):
    return "&lt;" in texte

print("ham avec &lt; :", df_hams["text"].map(contient_lt).sum())
print("spams avec &lt; :", df_spams["text"].map(contient_lt).sum())
display(df_clean[df_clean["text"].map(contient_lt)].head(5)[["text"]])

🆕 **Bonus facultatif (hors cours) : les mots les plus fréquents**

`df_spams["text"].str.lower().str.split().explode().value_counts().head(20)` enchaîne 4 outils : tout en minuscules → chaque texte devient une liste de mots → `explode` met un mot par ligne → on compte. Lance-le sur les spams puis sur les ham, et compare les listes.

In [ ]:
# 🆕 Bonus : les 20 mots les plus fréquents des spams, puis des ham
display(df_spams["text"].str.lower().str.split().explode().value_counts().head(20))
display(df_hams["text"].str.lower().str.split().explode().value_counts().head(20))

✍️ **À écrire (2-3 phrases)** : ce qui distingue un spam d'après tes lectures, et ce que tu fais de `&lt;#&gt;` (le garder tel quel, le remplacer ?).

*Ta réponse :* les spams sont plus longs (25 mots contre 11) et contiennent des appels à l'action (« call », « txt », « free », « claim », « prize »), des montants, des numéros courts ou surtaxés et plus de majuscules. `&lt;#&gt;` est un **nombre anonymisé** encodé en HTML (`&lt;` = `<`, `&gt;` = `>`), et les balises `&lt;…&gt;` n'apparaissent **que dans des ham** (222 messages, dont 197 `&lt;#&gt;`) : c'est un artefact de la collecte, pas un vrai signal. On le **garde tel quel** dans les CSV pour ne pas changer les données de référence, mais on le signale dans le livrable comme biais possible (le modèle peut apprendre « `&lt;#&gt;` ⇒ ham », ce qui n'existera pas dans les vrais SMS d'AT&T).

---
# Étape 4 — Séparer train / validation / test

🎯 Trois paquets, avec **la même part de spams (12,63 %)** dans chacun :

```
df_clean : 5 169 SMS
┌──────────────────── 70 % train ────────────────────┬── 15 % val ──┬── 15 % test ──┐
│ le modèle apprend dessus                            │ on règle :   │ on ne le      │
│                                                     │ époques,     │ regarde qu'à  │
│                                                     │ seuil...     │ la toute fin  │
└─────────────────────────────────────────────────────┴──────────────┴───────────────┘
```

`train_test_split` ne coupe qu'en **deux** : il faut l'appeler **deux fois**.
1. couper `df_clean` en `train_val_df` (85 %) et `test_df` (15 %) ;
2. couper `train_val_df` en `train_df` et `val_df`. Attention : 15 % du total = **15 / 85** de `train_val_df`.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `train_test_split(df, ...)` | coupe un DataFrame en deux | `test_size=`, `random_state=SEED` | 📍 M04-D01 Preprocessing with sklearn |
| `stratify=df["label"]` | garde la même part de chaque classe des deux côtés | la colonne des classes | 📍 M04-D04 Decision trees template, M05-D03 Neural networks with PyTorch |

Les colonnes `n_car` et `n_mots` peuvent partir avec : elles ne gênent pas.

In [ ]:
train_val_df, test_df = train_test_split(
    df_clean, test_size=0.15, stratify=df_clean["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=15 / 85, stratify=train_val_df["label"], random_state=SEED
)
for nom, paquet in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nom:5} : {len(paquet):4} SMS, {paquet['label'].mean():.2%} de spams")

In [ ]:
# ✅ Contrôle
assert (len(train_df), len(val_df), len(test_df)) == (3617, 776, 776), (len(train_df), len(val_df), len(test_df))
for nom, paquet in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert round(paquet["label"].mean(), 4) == 0.1263, f"{nom} : stratification ratée"
assert not set(train_df["text"]) & set(test_df["text"]), "un même SMS est dans train ET test"
print("✅ étape 4")

---
# Étape 5 — Enregistrer pour les phases suivantes

🎯 Écrire les trois paquets dans `src/train.csv`, `src/val.csv`, `src/test.csv`, avec seulement `text` et `label`. Les templates 2 et 3 partiront de ces fichiers.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `.to_csv(chemin, index=False)` | écrit un CSV | `index=False` : sinon une colonne de numéros en plus | 📍 M02-D01 Data Generation |
| `for nom, paquet in [("train", train_df), ...]:` | une boucle sur des paires (nom, DataFrame) | — | 🐍 Python (M01) |

Ces fichiers sont écrits en UTF-8 (le défaut de `to_csv`) : plus besoin de `encoding=` pour les relire.

In [ ]:
for nom, paquet in [("train", train_df), ("val", val_df), ("test", test_df)]:
    paquet[["text", "label"]].to_csv(DOSSIER_SRC / f"{nom}.csv", index=False)
    print("écrit :", DOSSIER_SRC / f"{nom}.csv")

In [ ]:
# ✅ Contrôle
for nom, n in [("train", 3617), ("val", 776), ("test", 776)]:
    relu = pd.read_csv(DOSSIER_SRC / f"{nom}.csv")
    assert list(relu.columns) == ["text", "label"], f"{nom}.csv : colonnes {list(relu.columns)}"
    assert len(relu) == n, f"{nom}.csv : {len(relu)} lignes au lieu de {n}"
print("✅ étape 5 : données prêtes pour les templates 2 et 3")

---
# 📝 Pour le livrable

Le notebook final doit **expliquer** le preprocessing. Garde de côté, en une ligne chacun :

| Décision | Ta justification |
|---|---|
| encodage latin-1 | le fichier n'est pas en UTF-8 : `read_csv` plante sans `encoding="latin-1"` |
| 50 messages recollés | des guillemets mal fermés ont coupé ces SMS à chaque virgule : on recolle la fin au lieu de la perdre |
| 403 doublons supprimés avant le split | un même SMS dans train et test gonflerait les scores (le modèle aurait déjà vu la réponse) |
| métrique principale (pas l'accuracy) | 12,6 % de spams : « toujours ham » fait 87 % d'accuracy, donc on suit le F1 et le recall de la classe spam |
| `max_len` choisi | 40 mots : couvre plus de 95 % des SMS (95 % = 33 mots) sans s'aligner sur le message de 171 mots |
| 70 / 15 / 15 stratifié | train pour apprendre, val pour régler, test pour la note finale ; même part de spams (12,63 %) dans les trois |
| `&lt;#&gt;` | nombre anonymisé, présent seulement dans des ham : gardé tel quel, signalé comme biais possible du dataset |